# Phase 5 — Feature Engineering

## What we are trying to accomplish

A feature is a concise numerical description extracted from measured data. For PHM, a useful feature should have a reason to respond to the physical degradation mechanism, be measurable reliably, and be comparable across observations. This phase extracts **candidate descriptors** from each captured switching transient so we can inspect them; it does not decide that any descriptor is a Health Indicator (HI).

The aging MAT files actually contain four time-domain transient channels: `gateSignalVoltage`, `gateEmitterVoltage`, `collectorEmitterVoltage`, and `collectorEmitterCurrentSignal`. Their values are sampled at a per-capture `dt`. Each capture is a separate short switching waveform, not a continuous row from the slow steady-state table. We retain the device folder, source file, capture number, timestamp, and `dt` with every feature row.

**Why these features:** mean, standard deviation, RMS, extrema, peak-to-peak, skewness, excess kurtosis, and crest factor summarize waveform offset, spread, amplitude, and shape. The dataset paper describes switching-transient voltage peak changes as a *candidate* indicator in a preliminary experiment. A whole-capture feature such as voltage peak-to-peak is **not the same** as a carefully isolated turn-off peak; it can also change with supply voltage, temperature set points, capture length, and measurement scaling. We will therefore label it a candidate descriptor, not a validated degradation feature.

**What we will not extract:** no FFT, dominant-frequency, or frequency-band feature: these records are short event captures, their sample intervals vary, and the frequency-domain arrays inspected for this experiment are empty. No cycle-count, learned embedding, RUL label, or HI is constructed. The README warns that transient records can be missing and collector-current measurements can drift; the features are exploratory, not calibrated physical quantities.

**Expected result:** one CSV row per actual transient capture, feature columns for the inspected waveform channels, retained provenance and sample metadata, plus a plot of a candidate voltage descriptor against capture time. The resulting table should be used to ask what varies and what confounders must be checked before any feature is trusted.

## 1. Locate and inspect the actual transient structure

MATLAB `.mat` files store nested structures, not ordinary spreadsheet columns. We check the discovered folder and an actual file before selecting channels. Each transient record stores its own `dt` and waveform arrays. Never assume that a sample index means the same duration when `dt` can differ.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import loadmat

project_root = Path.cwd()
if not (project_root / "data" / "raw").is_dir():
    project_root = project_root.parent

raw_root = project_root / "data" / "raw"
aging_folders = sorted(raw_root.rglob("Aging Data"))
if len(aging_folders) != 1:
    raise FileNotFoundError(
        f"Expected one 'Aging Data' directory under {raw_root}, found {len(aging_folders)}. "
        "Resolve which experiment to study before extracting features."
    )
aging_folder = aging_folders[0]
mat_files = sorted(aging_folder.rglob("*.mat"))
if not mat_files:
    raise FileNotFoundError(f"No MAT files found under {aging_folder}")

example_path = next(
    (path for path in mat_files if path.name.startswith("Device2 ")),
    mat_files[0],
)
example_mat = loadmat(example_path, squeeze_me=True, struct_as_record=False)
if "measurement" not in example_mat or not hasattr(example_mat["measurement"], "transient"):
    raise KeyError(f"No measurement.transient structure in {example_path.name}")

example_records = np.atleast_1d(example_mat["measurement"].transient)
example_record = example_records[0]
print("Example file:", example_path.name)
print("Transient captures in this file:", len(example_records))
print("Observed time-domain fields:", example_record.timeDomain._fieldnames)
print("dt:", float(np.asarray(example_record.timeDomain.dt).item()), "seconds")

## 2. Inspect one signal before choosing its features

Here we plot one actual collector-emitter-voltage transient. This channel is chosen because switching-voltage behavior is physically relevant to an IGBT, and the accompanying research paper discusses a voltage transient peak as a possible precursor in a preliminary experiment. That does **not** verify that a whole-capture statistic in this different multi-device experiment measures the same effect.

In [ ]:
channel_name = "collectorEmitterVoltage"
if channel_name not in example_record.timeDomain._fieldnames:
    raise KeyError(f"Selected record has no {channel_name!r} waveform")

example_dt = float(np.asarray(example_record.timeDomain.dt).item())
example_waveform = np.asarray(getattr(example_record.timeDomain, channel_name), dtype=float).squeeze()
if example_waveform.ndim != 1 or example_waveform.size < 2:
    raise ValueError(f"Expected a 1D waveform; got shape {example_waveform.shape}")
if not np.isfinite(example_waveform).all():
    raise ValueError("Example waveform contains missing or non-finite samples")

example_time_us = np.arange(example_waveform.size) * example_dt * 1e6
fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
ax.plot(example_time_us, example_waveform, linewidth=0.8)
ax.set(
    title=f"Observed {channel_name} capture ({example_path.name}, capture 0)",
    xlabel="Time since this capture began (µs)",
    ylabel="Recorded value (unit/calibration to verify)",
)
ax.grid(alpha=0.25)
feature_figures = project_root / "results" / "figures"
feature_figures.mkdir(parents=True, exist_ok=True)
example_plot_path = feature_figures / "feature_engineering_example_transient.png"
fig.savefig(example_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Samples: {example_waveform.size:,}; dt: {example_dt * 1e9:.1f} ns")
print(f"Saved: {example_plot_path.relative_to(project_root)}")

### Which time-domain features make sense to inspect?

- **Mean** describes the capture's average offset. It can be dominated by the amount of time the switching signal spends in each state.
- **Standard deviation** describes spread around the mean. **Variance** is standard deviation squared; it adds no independent information, so we do not save both.
- **RMS** is root-mean-square amplitude, including any DC offset. Do not confuse it with the AC-only RMS around a removed mean.
- **Minimum / maximum, absolute peak, and peak-to-peak** describe amplitude extremes. They may be relevant to electrical overstress but are sensitive to outliers, settings, capture length, and measurement scale.
- **Skewness** describes asymmetry; **excess kurtosis** describes tail weight relative to a normal distribution. They are generic shape summaries and are not automatically degradation mechanisms.
- **Crest factor** is absolute peak divided by RMS. It compares an extreme with the signal's overall RMS, but is sensitive to a single outlier and is undefined for zero RMS.

These summaries are calculated over one entire capture for teaching and screening. Because a capture may include switching plateaus and more than one edge, they are not turn-on/off-specific measurements. A defensible switching-time feature would require validated edge detection, event alignment, and matched operating conditions. Skewness and excess kurtosis are reported as missing for effectively constant channels because shape statistics are not meaningful there.

## 3. Calculate transparent time-domain summary statistics

`extract_time_domain_features` is a small reusable helper in `src/features.py`. It takes one measured waveform, checks that it is one-dimensional and finite, and returns explicit sample statistics. SciPy's `skew` and `kurtosis` implement shape statistics; `fisher=True` reports **excess kurtosis**, for which a normal distribution has value 0. The helper does not normalize, filter, window, or label the waveform.

In [ ]:
from src.features import extract_time_domain_features

example_features = extract_time_domain_features(example_waveform)
pd.Series(example_features, name=channel_name)

RMS is calculated on the recorded signal as stored; no offset is removed, because that would alter its meaning. We use an absolute peak to define crest factor consistently even if a recorded channel has a negative excursion. These are descriptive statistics in the waveform's recorded numerical scale—not calibrated engineering units.

## 4. Which signals are descriptive, operating-context, or candidate signals?

- `gateSignalVoltage` is the recorded gate command; it is useful context for what was applied.
- `gateEmitterVoltage` describes the measured gate-emitter response and may reveal differences in drive/capture conditions.
- `collectorEmitterVoltage` describes the switching voltage response; peak-related descriptions are a physically motivated **candidate** to inspect, not a selected HI.
- `collectorEmitterCurrentSignal` describes the switching current, but the documentation specifically warns of about 600 mA drift. Treat current summaries cautiously until their calibration is resolved.

Steady-state `packageTemperature` and voltage/current columns are slow measurements, not the same waveforms as the transient arrays. They are not combined here with the fast capture features. The documented SMU threshold, breakdown, and leakage characterizations could be domain-relevant only if their test protocol and mapping to each device's aging history are established; they are not mixed into this transient table.

## 5. Extract features for every observed capture, keeping provenance

Each feature row represents one transient capture from one source file. It carries capture index, recorded date/time, `dt`, derived sample rate, sample count, and capture duration. Because `dt` varies across captures, retaining it is essential. The captures named `check` are retained too, but a filename is not taken as proof of a health state. Any missing or malformed waveform stops with a descriptive error rather than silently producing incomplete features.

In [ ]:
waveform_channels = list(example_record.timeDomain._fieldnames)
if "dt" not in waveform_channels:
    raise KeyError("Transient timeDomain fields do not include 'dt'")
waveform_channels.remove("dt")
if not waveform_channels:
    raise ValueError("No transient waveform channels were found")

feature_rows = []
for mat_path in mat_files:
    contents = loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    if "measurement" not in contents or not hasattr(contents["measurement"], "transient"):
        raise KeyError(f"Expected measurement.transient in {mat_path}")

    records = np.atleast_1d(contents["measurement"].transient)
    for capture_index, capture in enumerate(records):
        time_domain = capture.timeDomain
        capture_fields = list(time_domain._fieldnames)
        if "dt" not in capture_fields:
            raise KeyError(f"No dt in {mat_path.name}, capture {capture_index}")
        if set(capture_fields) != set(waveform_channels + ["dt"]):
            raise ValueError(
                f"Unexpected transient fields in {mat_path.name}, capture {capture_index}: "
                f"{capture_fields}"
            )

        dt = float(np.asarray(time_domain.dt).item())
        if not np.isfinite(dt) or dt <= 0:
            raise ValueError(f"Invalid dt in {mat_path.name}, capture {capture_index}: {dt}")

        row = {
            "device_folder": mat_path.parent.name,
            "source_file": str(mat_path.relative_to(aging_folder)),
            "capture_index": capture_index,
            "date": str(capture.date),
            "timeEpoch": float(np.asarray(capture.timeEpoch).item()),
            "dt_seconds": dt,
            "sample_rate_hz": 1 / dt,
        }
        sample_counts = set()
        for channel in waveform_channels:
            values = np.asarray(getattr(time_domain, channel), dtype=float).squeeze()
            if values.ndim != 1 or values.size < 2:
                raise ValueError(
                    f"Expected a 1D waveform in {mat_path.name}, capture {capture_index}, "
                    f"channel {channel}; got shape {values.shape}"
                )
            if not np.isfinite(values).all():
                raise ValueError(
                    f"Non-finite samples in {mat_path.name}, capture {capture_index}, channel {channel}"
                )
            sample_counts.add(values.size)
            channel_features = extract_time_domain_features(values)
            for name, value in channel_features.items():
                if name != "sample_count":
                    row[f"{channel}_{name}"] = value

        if len(sample_counts) != 1:
            raise ValueError(
                f"Waveform channels have different lengths in {mat_path.name}, capture {capture_index}: "
                f"{sorted(sample_counts)}"
            )
        sample_count = sample_counts.pop()
        row["sample_count"] = sample_count
        row["duration_seconds"] = (sample_count - 1) * dt
        row["nyquist_hz"] = (1 / dt) / 2
        feature_rows.append(row)

transient_features = pd.DataFrame(feature_rows)
print(f"Feature rows: {len(transient_features):,}")
print(f"Device folders: {transient_features['device_folder'].nunique()}")
print(f"Source MAT files: {transient_features['source_file'].nunique()}")
print(f"Waveform channels: {waveform_channels}")
print(f"Feature-table shape: {transient_features.shape}")

## 6. Inspect the feature table and compare acquisition settings

The columns have a common pattern: `channel_feature`. For example, `collectorEmitterVoltage_peak_to_peak` summarizes the minimum-to-maximum range in that entire recorded capture. It is a voltage **descriptor**, not a calibrated switching overshoot feature. This `head`, `shape`, and summary inspection is now applied to the extracted feature table—not to the raw waveforms themselves.

In [ ]:
transient_features.head()

In [ ]:
transient_features.shape, transient_features["source_file"].value_counts().sort_index()

In [ ]:
metadata_columns = [
    "device_folder", "source_file", "capture_index", "date", "timeEpoch",
    "dt_seconds", "sample_rate_hz", "sample_count", "duration_seconds", "nyquist_hz",
]
feature_columns = [column for column in transient_features.columns if column not in metadata_columns]
transient_features[metadata_columns].describe(include="all").T

In [ ]:
transient_features[feature_columns].describe().T

In [ ]:
acquisition_summary = (
    transient_features.groupby(["device_folder", "source_file"], observed=True)
    .agg(
        captures=("capture_index", "size"),
        distinct_sample_intervals_ns=("dt_seconds", lambda values: ", ".join(
            f"{value * 1e9:g}" for value in sorted(values.unique())
        )),
        sample_counts=("sample_count", lambda values: ", ".join(
            str(value) for value in sorted(values.unique())
        )),
        capture_duration_ms_min=("duration_seconds", lambda values: values.min() * 1e3),
        capture_duration_ms_max=("duration_seconds", lambda values: values.max() * 1e3),
    )
)
acquisition_summary

### Why acquisition metadata matters for features

Features from records with different `dt`, sample counts, or capture durations are not automatically comparable. Peak-to-peak and absolute extrema may change simply because a longer capture has more opportunity to contain an extreme value. Crest factor is especially sensitive to isolated peaks. Preserve acquisition metadata, and only compare matched operating points and acquisition settings unless a justified adjustment is developed.

## 7. Compare a candidate waveform descriptor over capture time

We plot `collectorEmitterVoltage_peak_to_peak` from the extracted table to see whether it varies across observed capture dates for each device. The main and `b` files stay separate by marker; low-temperature `check` files are shown separately. This whole-capture range is **not** the same as a turn-off peak, and changes can be caused by supply settings or temperature stage as well as device condition. Use the aging log to check conditions before interpreting any pattern.

In [ ]:
candidate_feature = "collectorEmitterVoltage_peak_to_peak"
if candidate_feature not in transient_features.columns:
    raise KeyError(f"Expected extracted candidate descriptor {candidate_feature!r}")

plot_features = transient_features.loc[
    np.isfinite(transient_features["timeEpoch"])
    & (transient_features["timeEpoch"] > 0)
].copy()
first_timestamp = plot_features.groupby("device_folder", observed=True)["timeEpoch"].transform("min")
plot_features["elapsed_recorded_hours"] = (plot_features["timeEpoch"] - first_timestamp) / 3600
plot_features["file_kind"] = plot_features["source_file"].map(
    lambda value: "check" if "check" in Path(value).stem.lower() else "main / continuation"
)

devices = sorted(plot_features["device_folder"].unique())
fig, axes = plt.subplots(
    nrows=len(devices), ncols=1, figsize=(11, 3 * len(devices)),
    squeeze=False, constrained_layout=True
)
for ax, device in zip(axes[:, 0], devices):
    device_data = plot_features.loc[plot_features["device_folder"] == device]
    for kind, marker, color in [
        ("main / continuation", "o", "tab:blue"),
        ("check", "x", "tab:orange"),
    ]:
        segment = device_data.loc[device_data["file_kind"] == kind]
        for source_file, file_data in segment.groupby("source_file", sort=True, observed=True):
            file_data = file_data.sort_values("timeEpoch")
            ax.scatter(
                file_data["elapsed_recorded_hours"],
                file_data[candidate_feature],
                s=20,
                marker=marker,
                color=color,
                label=Path(source_file).name,
            )
    ax.set_title(device)
    ax.set_ylabel("Recorded peak-to-peak (units unverified)")
    ax.grid(alpha=0.25)
    ax.legend(fontsize="x-small", ncol=2)
axes[-1, 0].set_xlabel("Elapsed recorded clock time (hours)")
fig.suptitle("Candidate descriptor over capture time — not a validated HI")

candidate_plot_path = feature_figures / "feature_engineering_candidate_peak_to_peak.png"
fig.savefig(candidate_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {candidate_plot_path.relative_to(project_root)}")

### What does this plot tell us about degradation?

It shows how a simple whole-capture voltage range varies across the recorded observations; it does **not** prove degradation or isolate the paper's turn-off peak. First check the aging log for stress settings at each capture, then ask whether similar shapes were acquired under comparable temperature, supply, gate, and sampling conditions. File gaps, changing `dt`/duration, measurement scale, and sample-to-sample variability are competing explanations. A promising physical hypothesis still needs a specifically aligned turn-off feature and measurement-validation evidence.

## 8. Why frequency-domain features are deferred

FFT magnitude, dominant frequency, spectral energy, and band energy should be calculated only for a question that supports frequency analysis. Across all transient captures, the named `frequencyDomain` voltage/current amplitude and phase arrays are empty; `dxAmplitude` and `dxPhase` are scalar metadata, not spectra. The raw time-domain captures are short switching events—not long, demonstrably stationary records—and `dt` varies by capture. An FFT could describe the edge shape, capture window, switching plateau, or measurement-chain response rather than damage progression. We therefore do **not** manufacture FFT values from the empty stored fields or make a spectral PHM feature here.

A later, well-scoped switching/ringing study could consider spectra from aligned, comparable event windows after checking `dt`, acquisition bandwidth, periodicity, capture length, operating settings, and transducer calibration. That is a separate experiment design decision, not an automatic step in this feature set.

## 9. Save candidate features outside the raw folder

The file contains descriptive measurements for every available transient capture and provenance needed to trace a row back to its original MAT file and capture. The output is saved to `data/features/`; raw MATLAB files are never modified. This feature table has no failure status, EOL time, RUL target, or selected HI.

In [ ]:
feature_folder = project_root / "data" / "features"
feature_folder.mkdir(parents=True, exist_ok=True)
feature_path = feature_folder / "transient_candidate_features.csv"
transient_features.to_csv(feature_path, index=False)
print(f"Saved {len(transient_features):,} capture rows to {feature_path.relative_to(project_root)}")
print("Features are uncalibrated exploratory descriptors; no RUL or EOL labels were created.")

## Phase 5 learning check

1. What does each extracted row represent? Which columns let you trace it to its source capture and understand its sample rate?
2. Why might peak-to-peak or crest factor change when acquisition length, supply voltage, temperature, or one noisy sample changes?
3. Why did we retain the gate-command signals as experimental context rather than automatically treating them as health measures?
4. Why must collector-emitter current descriptors be treated cautiously here?
5. What additional signal processing and experimental evidence are required before estimating an aligned turn-off peak across records?
6. What would make an extracted feature a validated Health Indicator rather than only a candidate descriptor?
7. Why did we defer FFT-derived features, and under what carefully controlled question might you revisit them?

Run the notebook in VS Code from top to bottom and share your observations before moving on. This phase extracts candidate features for study; it does not select an HI or define RUL.